In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import uuid
 
cat = 'capstone_project_07'
B = f'{cat}.schema_capstone07_bronze'
S = f'{cat}.schema_capstone07_silver'
run_id = str(uuid.uuid4())[:8]
OUTLIER_MS = 500          # response_ms above this is flagged (35 rows in your data)
 
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {S}')
 
# Reason text for every rule ID
REASONS = {
    'R01': 'mandatory field missing',
    'R02': 'timestamp not parseable',
    'R03': 'numeric field not parseable',
    'R05': 'exact duplicate removed, latest arrival kept',
    'R07': 'ingest time before event time',
    'R08': 'invalid access_status',
    'R09': 'document_id or user_id not in reference',
    'R10': 'invalid access_type value',
    'C01': 'CDC key, timestamp or sequence invalid',
    'C02': 'CDC event not in Silver (orphan)',
    'C03': 'CDC change dated before the event',
    'C04': 'invalid new_status',
    'C05': 'duplicate CDC change removed',
    'S01': 'secondary key missing',
    'S02': 'secondary timestamp invalid',
    'S03': 'secondary value not numeric',
    'S04': 'invalid event_type',
    'S05': 'orphan access_event_id',
    'S06': 'duplicate event_id removed',
}
reason_map = F.create_map(*[x for k, v in REASONS.items() for x in (F.lit(k), F.lit(v))])
 
 
def upsert(df, table, keys):
    """Create the empty table once, then MERGE so reruns never duplicate rows."""
    if not spark.catalog.tableExists(table):
        df.limit(0).write.saveAsTable(table)
    cond = ' AND '.join(f't.{k} = s.{k}' for k in keys)
    (DeltaTable.forName(spark, table).alias('t')
        .merge(df.alias('s'), cond)
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
 
 
def log_rejects(df, source_table, key, raw_cols):
    """Replace this source's rows in the rejected-records table with the current ones."""
    spark.sql('DELETE FROM ' + S + '.silver_rejected_records WHERE source_table = ' + repr(source_table))
    (df.select(F.lit(source_table).alias('source_table'),
               F.col(key).alias('record_key'),
               F.col('reject_rule').alias('rule_id'),
               reason_map.getItem(F.col('reject_rule')).alias('reason'),
               F.to_json(F.struct(*raw_cols)).alias('raw_record'),
               F.col('source_file'),
               F.current_timestamp().alias('rejected_at'),
               F.lit(run_id).alias('run_id'))
       .write.mode('append').saveAsTable(f'{S}.silver_rejected_records'))
 
 
def audit(step, rin, rout, rrej, rdup, rflag, note):
    (spark.createDataFrame([(run_id, step, rin, rout, rrej, rdup, rflag, note)],
        'run_id string, step string, rows_in long, rows_out long, rows_rejected long, '
        'rows_duplicates long, rows_flagged long, note string')
     .withColumn('logged_at', F.current_timestamp())
     .write.mode('append').saveAsTable(f'{S}.silver_dq_audit'))
 
 
print('setup done, run_id =', run_id)
 

In [0]:
print('run_id:', run_id)
print('helpers defined:', callable(upsert), callable(log_rejects), callable(audit))
display(spark.sql('SHOW SCHEMAS IN capstone_project_07'))

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {S}.silver_rejected_records (
  source_table STRING, record_key STRING, rule_id STRING, reason STRING,
  raw_record STRING, source_file STRING, rejected_at TIMESTAMP, run_id STRING)
""")
 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {S}.silver_dq_audit (
  run_id STRING, step STRING, rows_in BIGINT, rows_out BIGINT, rows_rejected BIGINT,
  rows_duplicates BIGINT, rows_flagged BIGINT, note STRING, logged_at TIMESTAMP)
""")
 
display(spark.sql(f'SHOW TABLES IN {S}'))

In [0]:
# Rename the Bronze reference tables if that has not been done yet
for old, new in [('`document`', 'bronze_document_id'), ('`user`', 'bronze_user_id')]:
    if spark.catalog.tableExists(f'{B}.{old}') and not spark.catalog.tableExists(f'{B}.{new}'):
        spark.sql(f'ALTER TABLE {B}.{old} RENAME TO {B}.{new}')
 
# Documents: one row per document version (SCD2 structure)
wd = Window.partitionBy('document_id').orderBy('valid_from')
doc_v = (spark.table(f'{B}.bronze_document_id')
    .select('document_id', 'display_name', 'region',
            F.col('active_flag').alias('doc_active_flag'),
            F.col('risk_tier').alias('doc_risk_tier'),
            F.expr('try_cast(effective_date as date)').alias('valid_from'))
    .dropDuplicates(['document_id', 'valid_from'])
    .withColumn('version_no', F.row_number().over(wd))
    .withColumn('valid_to', F.lead('valid_from').over(wd))
    .withColumn('is_current', F.col('valid_to').isNull()))
doc_v.write.mode('overwrite').saveAsTable(f'{S}.silver_document_versions')
 
# Users
usr = (spark.table(f'{B}.bronze_user_id')
    .select('user_id', 'display_name', 'region',
            F.col('active_flag').alias('user_active_flag'),
            F.col('risk_tier').alias('user_risk_tier'),
            F.expr('try_cast(effective_date as date)').alias('valid_from'))
    .dropDuplicates(['user_id']))
usr.write.mode('overwrite').saveAsTable(f'{S}.silver_user')
 
print('documents:', spark.table(f'{S}.silver_document_versions').count(),
      '| users:', spark.table(f'{S}.silver_user').count())
 

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS docs, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_docs,
       MAX(version_no) AS max_version,
       SUM(CASE WHEN doc_active_flag = 'N' THEN 1 ELSE 0 END) AS inactive_docs
FROM {S}.silver_document_versions
"""))
 

In [0]:
STATUS_OK = ['OPEN', 'PENDING', 'COMPLETED', 'FAILED']
TYPE_OK = ['SEARCH', 'VIEW', 'DOWNLOAD']
 
 
def check_events(raw, docs, users):
    # try_ functions return NULL on bad values instead of crashing (serverless uses ANSI mode)
    t = (raw
         .withColumn('event_ts', F.expr('try_to_timestamp(event_timestamp)'))
         .withColumn('ingest_ts', F.expr('try_to_timestamp(ingest_timestamp)'))
         .withColumn('response_ms_num', F.expr('try_cast(response_ms as double)'))
         .withColumn('relevance_num', F.expr('try_cast(relevance_score as double)')))
    t = (t.join(docs.select('document_id').distinct().withColumn('doc_ok', F.lit(1)), 'document_id', 'left')
          .join(users.select('user_id').distinct().withColumn('usr_ok', F.lit(1)), 'user_id', 'left'))
    missing = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == '')
    return t.withColumn(
        'reject_rule',
        F.when(missing('access_event_id') | missing('document_id') | missing('user_id')
               | missing('event_timestamp'), 'R01')
         .when(F.col('event_ts').isNull() | F.col('ingest_ts').isNull(), 'R02')
         .when(F.col('response_ms_num').isNull() | F.col('relevance_num').isNull(), 'R03')
         .when(F.col('ingest_ts') < F.col('event_ts'), 'R07')
         .when(F.col('access_status').isNull() | ~F.col('access_status').isin(STATUS_OK), 'R08')
         .when(F.col('access_type').isNotNull() & ~F.col('access_type').isin(TYPE_OK), 'R10')
         .when(F.col('doc_ok').isNull() | F.col('usr_ok').isNull(), 'R09'))
 
 
print('rule function defined')
 

In [0]:
def bronze(name):
    """Read a Bronze table and give its load-time column the name 'ingested_at'."""
    df = spark.table(f'{B}.{name}')
    if 'bronze_ingestion_timestamp' in df.columns and 'ingested_at' not in df.columns:
        df = df.withColumnRenamed('bronze_ingestion_timestamp', 'ingested_at')
    return df
 
spark.sql(f'DROP TABLE IF EXISTS {S}.silver_access_events')   # rebuilt from Bronze, safe to drop
 
raw0 = bronze('bronze_access_events')
raw_cols = raw0.columns
raw = raw0
if 'source_schema_version' not in raw.columns:
    raw = raw.withColumn('source_schema_version', F.lit(None).cast('string'))
 
checked = check_events(raw, bronze('bronze_document_id'), bronze('bronze_user_id'))
 
rej = checked.filter('reject_rule IS NOT NULL')
valid = checked.filter('reject_rule IS NULL')
 
# Duplicates: keep the latest arrival for each access_event_id
w = Window.partitionBy('access_event_id').orderBy(
    F.col('ingest_ts').desc(), F.col('source_file').desc(), F.col('ingested_at').desc())
ranked = valid.withColumn('rn', F.row_number().over(w))
dups = ranked.filter('rn > 1').withColumn('reject_rule', F.lit('R05'))
dedup = ranked.filter('rn = 1').drop('rn')
 
docs_s = (spark.table(f'{S}.silver_document_versions').filter('is_current')
          .select('document_id', 'doc_active_flag', 'doc_risk_tier'))
users_s = spark.table(f'{S}.silver_user').select('user_id', 'user_active_flag', 'user_risk_tier')
 
clean = (dedup
    .withColumn('access_type_clean', F.coalesce(F.col('access_type'), F.lit('UNKNOWN')))
    .withColumn('is_access_type_missing', F.col('access_type').isNull())
    .withColumn('is_outlier', F.col('response_ms_num') > OUTLIER_MS)
    .join(docs_s, 'document_id', 'left')
    .join(users_s, 'user_id', 'left')
    .select('access_event_id', 'event_ts', 'ingest_ts', 'document_id', 'user_id', 'workspace_id',
            'access_type', 'access_type_clean', 'is_access_type_missing', 'access_status',
            F.col('response_ms_num').alias('response_ms'),
            F.col('relevance_num').alias('relevance_score'),
            'is_outlier', 'content_class', 'source_system', 'source_schema_version',
            'doc_active_flag', 'doc_risk_tier', 'user_active_flag', 'user_risk_tier',
            'source_file', 'ingested_at'))
 
upsert(clean, f'{S}.silver_access_events', ['access_event_id'])
log_rejects(rej.unionByName(dups.select(*rej.columns)), 'bronze_access_events', 'access_event_id', raw_cols)
 
n_in, n_rej, n_dup, n_out = raw0.count(), rej.count(), dups.count(), clean.count()
n_flag = clean.filter('is_outlier OR is_access_type_missing').count()
print(f'in={n_in} rejected={n_rej} duplicates={n_dup} out={n_out} flagged={n_flag}')
assert n_in == n_rej + n_dup + n_out, 'counts do not reconcile'
audit('access_events', n_in, n_out, n_rej, n_dup, n_flag, f'outlier>{OUTLIER_MS}; UNKNOWN access_type')
 

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS rows_total, COUNT(DISTINCT access_event_id) AS distinct_ids,
       SUM(CASE WHEN is_outlier THEN 1 ELSE 0 END) AS outliers,
       SUM(CASE WHEN is_access_type_missing THEN 1 ELSE 0 END) AS unknown_type
FROM {S}.silver_access_events
"""))
display(spark.sql(f'SELECT rule_id, reason, COUNT(*) AS n FROM {S}.silver_rejected_records GROUP BY 1, 2'))
display(spark.sql(f'SELECT access_status, COUNT(*) AS n FROM {S}.silver_access_events GROUP BY 1 ORDER BY 1'))
display(spark.sql(f'SELECT * FROM {S}.silver_dq_audit ORDER BY logged_at'))
 

In [0]:
CDC_OK = ['OPEN', 'PENDING', 'COMPLETED', 'FAILED', 'CANCELLED', 'REVERSED']
raw_cdc = bronze('bronze_status_cdc')
ev = spark.table(f'{S}.silver_access_events').select(
    'access_event_id', F.col('access_status').alias('event_status'), F.col('event_ts').alias('ev_ts'))
 
c = (raw_cdc
     .withColumn('change_ts', F.expr('try_to_timestamp(change_timestamp)'))
     .withColumn('seq', F.expr('try_cast(source_sequence as int)'))
     .join(ev, 'access_event_id', 'left'))
 
c = c.withColumn(
    'reject_rule',
    F.when(F.col('access_event_id').isNull() | F.col('change_ts').isNull() | F.col('seq').isNull(), 'C01')
     .when(F.col('ev_ts').isNull(), 'C02')
     .when(F.col('change_ts') < F.col('ev_ts'), 'C03')
     .when(F.col('new_status').isNull() | ~F.col('new_status').isin(CDC_OK), 'C04'))
 
crej = c.filter('reject_rule IS NOT NULL')
cvalid = c.filter('reject_rule IS NULL')
 
wc = Window.partitionBy('access_event_id', 'seq').orderBy(F.col('ingested_at').desc())
cr = cvalid.withColumn('rn', F.row_number().over(wc))
cdup = cr.filter('rn > 1').withColumn('reject_rule', F.lit('C05'))
chg = cr.filter('rn = 1').drop('rn')
 
# The first change of an event must start from the status the event had. Record any mismatch.
wf = Window.partitionBy('access_event_id')
hist = (chg
    .withColumn('is_first', F.col('seq') == F.min('seq').over(wf))
    .withColumn('prev_status_mismatch',
                F.col('is_first') & (F.col('previous_status') != F.col('event_status')))
    .select('access_event_id', 'change_ts', 'seq', 'previous_status', 'new_status', 'change_reason',
            'prev_status_mismatch', 'source_system', 'source_file', 'ingested_at'))
 
upsert(hist, f'{S}.silver_status_history', ['access_event_id', 'seq'])
log_rejects(crej.unionByName(cdup.select(*crej.columns)), 'bronze_status_cdc', 'access_event_id', raw_cdc.columns)
 
n_c_in, n_c_rej, n_c_dup, n_c_out = raw_cdc.count(), crej.count(), cdup.count(), hist.count()
print(f'cdc in={n_c_in} rejected={n_c_rej} duplicates={n_c_dup} history={n_c_out}')
assert n_c_in == n_c_rej + n_c_dup + n_c_out, 'cdc counts do not reconcile'
audit('status_cdc', n_c_in, n_c_out, n_c_rej, n_c_dup, 0, 'latest change wins')
 

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS history_rows,
       SUM(CASE WHEN prev_status_mismatch THEN 1 ELSE 0 END) AS mismatches
FROM {S}.silver_status_history
"""))
display(spark.sql(f'SELECT new_status, COUNT(*) AS n FROM {S}.silver_status_history GROUP BY 1 ORDER BY 1'))